# Stage 1: Per-Wallet Copy Sizing (tier3@2-0)

Fit per-wallet copy weights ``alpha_w`` on **train** per-wallet daily pnl
(3 tiers by Sharpe proxy: top 2x, middle 1x, bottom dropped), pick
hyperparameters on **validation** by sim Sharpe, single **test** pass.
Copy qty is capped by the reconstructed share-depth ``bucket_avail_copy_qty``.

**Output:** `stage1_scaled_result.json` + `signal_lab/wallet_scaling_{sim,ci,contrib}.csv`


In [ ]:
# from signal_lab.depth_cap import clear_cache
# clear_cache()  # uncomment to force rebuild of data/bucket_depth.parquet

cleared depth lookup cache: /Users/vobornij/projects/polymarket/data/bucket_depth.parquet


In [26]:
# Setup: imports, paths, constants
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

NB_DIR = Path.cwd() if "__file__" not in globals() else Path(__file__).resolve().parent
sys.path.insert(0, str(NB_DIR))
OUT_DIR = NB_DIR / "signal_lab"

import numpy as np
import pandas as pd

from lib import DEFAULT_TAGS
from signal_lab.depth_cap import build_lookup, clear_cache
from signal_lab.filters import COPY_DEFAULT
from signal_lab.signal_lib import spearman_rho
from signal_lab.sizing import (
    block_bootstrap_sharpe,
    capital_constrained_sim,
    sizing_sharpe,
)
from signal_lab.stage1 import candidate_splits_for, load_stage1_data
from signal_lab.wallet_scaling import (
    alpha_kelly,
    alpha_tier,
    attach_depth_cap,
    run_sim,
    sim_row,
    wallet_daily_pnl,
    wallet_stats,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

BUDGET = 10_000.0
COST_SEL = 10.0
ALPHA_MAX_GRID = (2.0, 4.0, 8.0)
TIER_GRID = [(nt, am, amin) for nt in (3, 4, 5) for am in ALPHA_MAX_GRID for amin in (0.0, 0.25)]
UNIFORM_K_GRID = (0.5, 1.0, 2.0, 4.0)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load data

In [27]:
df_full, df_train, df_val, df_test, wallet_metrics, hold_metrics = load_stage1_data()
print(f"df_full: {len(df_full):,}")
print(f"  train: {len(df_train):,}  val: {len(df_val):,}  test: {len(df_test):,}")


Markets: 2165960
Filtered markets for {'Politics'}: 44273
Loading 16 trade shards...
Total trades loaded: 13,282,697
Unique wallets: 34,344
Date range: 2025-01-01 00:00:59+00:00 -> 2026-08-03 16:01:48+00:00
Chronological split: train <= 2025-09-15T00:00:00Z, val <= 2026-03-10T00:00:00Z, test > 2026-03-10T00:00:00Z
Method: chronological  |  Unique end dates: 518  (train=207, val=155, test=156)

  Train:  1,539,251 trades  (3,276 markets)
  Val:    4,250,479 trades  (6,136 markets)
  Test:   7,492,967 trades  (11,639 markets)
  Total: 13,282,697 trades  (21,051 markets)


/Users/vobornij/projects/polymarket/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


df_full: 13,282,697
  train: 1,539,251  val: 4,250,479  test: 7,492,967


## Copy universe

Candidate wallets = `COPY_DEFAULT` (copy-default filter).

In [28]:
wallets = set(COPY_DEFAULT(wallet_metrics, hold_metrics))
print(f"copy_default wallets: {len(wallets)}")


copy_default wallets: 45


## Reconstruct share-depth cap

Load the cached `bucket_avail_copy_qty` lookup per (tx_hash, wallet, side, token_id); rebuilt from the enriched shards on first run or after `clear_cache()`.

In [29]:
lookup = build_lookup()
print(f"depth lookup buckets: {len(lookup):,}")
lookup["bucket_avail_copy_qty"].describe(percentiles=[.5, .9, .99])


  [1/16] enriched_0.parquet: 5,899,740 rows
  [2/16] enriched_1.parquet: 6,137,763 rows
  [3/16] enriched_2.parquet: 6,655,869 rows
  [4/16] enriched_3.parquet: 6,728,549 rows
  [5/16] enriched_4.parquet: 6,859,909 rows
  [6/16] enriched_5.parquet: 6,605,170 rows
  [7/16] enriched_6.parquet: 7,216,838 rows
  [8/16] enriched_7.parquet: 6,669,464 rows
  [9/16] enriched_8.parquet: 6,506,480 rows
  [10/16] enriched_9.parquet: 6,039,978 rows
  [11/16] enriched_a.parquet: 7,032,284 rows
  [12/16] enriched_b.parquet: 6,692,404 rows
  [13/16] enriched_c.parquet: 6,392,674 rows
  [14/16] enriched_d.parquet: 6,752,824 rows
  [15/16] enriched_e.parquet: 6,000,977 rows
  [16/16] enriched_f.parquet: 6,222,848 rows
wrote /Users/vobornij/projects/polymarket/data/bucket_depth.parquet: 81,254,995 buckets
depth lookup buckets: 81,254,995


count   81254995.0000
mean        2155.3274
std        31900.1855
min           -0.0000
50%            0.0000
90%          761.4000
99%        34747.5977
max     10237992.0000
Name: bucket_avail_copy_qty, dtype: float64

In [30]:
splits = candidate_splits_for(df_full, wallets)
splits = attach_depth_cap(splits, wallets, lookup=lookup)
del df_full, df_train, df_val, df_test, lookup

for name in ("train", "val", "test"):
    fr = splits[name]
    capped = (fr["bucket_avail_copy_qty"] < fr["copyable_qty"]).mean()
    print(f"{name:5s}: {len(fr):,}  trades_capped_by_depth={capped:.3f}")


Chronological split: train <= 2025-09-19T00:00:00Z, val <= 2026-03-02T00:00:00Z, test > 2026-03-02T00:00:00Z
Method: chronological  |  Unique end dates: 456  (train=182, val=136, test=138)

  Train:     20,252 trades  (1,994 markets)
  Val:       21,303 trades  (2,464 markets)
  Test:       9,339 trades  (1,606 markets)
  Total:     50,894 trades  (6,064 markets)
lookup rows (candidate wallets only): 107,859
train: 20,252  trades_capped_by_depth=0.000
val  : 21,303  trades_capped_by_depth=0.000
test : 9,339  trades_capped_by_depth=0.000


## Train per-wallet stats

Per-wallet daily pnl (copyable, alpha=1) with mean/std shrinkage -> Sharpe proxy.

In [31]:
train_daily = wallet_daily_pnl(splits["train"])
st = wallet_stats(train_daily)
print(f"wallets with train daily series: {len(st)}")
st[["mu", "sigma", "n_days", "sharpe_proxy", "total_pnl"]].sort_values(
    "sharpe_proxy", ascending=False
).head(15)


wallets with train daily series: 45


,mu,sigma,n_days,sharpe_proxy,total_pnl
wallet,,,,,
0x183a2a0b034877e761af0778da5134bebc37d514,3758.7952,10574.7140,10,0.2059,37587.9521
0xf72044fb3f98854411a069d9a682e54e4b824599,380.1124,2021.1009,36,0.1334,13684.0476
0xc5fe4e2927af431482ae4bb820b9e122c7c4e320,329.3853,781.7847,16,0.1161,5270.1650
0x89c73a6b97b2e58972fb83b7d661121814ba3ff0,266.6527,969.9679,22,0.1108,5866.3605
0x70c907411393dd590ef46908a80bb4cfb3dfcef3,218.4473,687.2109,27,0.1093,5898.0758
0x2625e0f62524fea18c26edd95877ce1710c8b2f8,246.6406,585.3277,16,0.0958,3946.2495
0xd3989ba133ab48b5b3a81e3dba9b37b5966a46d7,139.7143,1199.3943,106,0.0944,14809.7197
0x57c5de7efafb020e589f80e0da6e16e9b5c907aa,140.1978,723.7624,46,0.0938,6449.0966
0xf59fe344fb8af90a14bd6f2fe62430e56dc03be9,112.0526,592.5222,67,0.0928,7507.5252


## Weight schemes

All benchmarked vs copy-all: shrunk max-Sharpe (Kelly), tier, uniform-k.

In [32]:
schemes = {}
for am in ALPHA_MAX_GRID:
    schemes[f"kelly@{am:g}"] = ("kelly", alpha_kelly(st, am), {"alpha_max": am})
for (nt, am, amin) in TIER_GRID:
    schemes[f"tier{nt}@{am:g}-{amin:g}"] = (
        "tier",
        alpha_tier(st, nt, am, amin),
        {"n_tiers": nt, "alpha_max": am, "alpha_min": amin},
    )
for k in UNIFORM_K_GRID:
    schemes[f"uniform@{k:g}"] = ("uniform", pd.Series(k, index=st.index), {"k": k})
schemes["copy_all"] = ("copy_all", pd.Series(1.0, index=st.index), {})

print(f"schemes: {len(schemes)}")


schemes: 26


## Validation grid search

Objective: annualized Sharpe of daily resolution-pnl, fixed $10k budget, 10bps.

In [33]:
sim_rows = []
best_per_scheme = {}
for name, (scheme, alpha_map, params) in schemes.items():
    res = run_sim(splits["val"], alpha_map, COST_SEL)
    row = sim_row(scheme, name, "val", res)
    sim_rows.append(row)
    key = scheme if scheme != "kelly" else "kelly"
    if key not in best_per_scheme or row["sharpe_daily"] > best_per_scheme[key][2]:
        best_per_scheme[key] = (name, params, row["sharpe_daily"])

sim_df = pd.DataFrame(sim_rows)
sim_df[sim_df["split"] == "val"].sort_values("sharpe_daily", ascending=False).head(15)


,scheme,config,split,trades,pnl,roi_w,sharpe_daily,mean_used,peak_used
21,uniform,uniform@0.5,val,7752,20988.4500,0.2336,1.5760,7364.7200,10000.0000
3,tier,tier3@2-0,val,4731,26165.2700,0.3292,1.5740,8156.2000,10000.0000
5,tier,tier3@4-0,val,4731,26165.2700,0.3292,1.5740,8156.2000,10000.0000
7,tier,tier3@8-0,val,4731,26165.2700,0.3292,1.5740,8156.2000,10000.0000
19,tier,tier5@8-0,val,5463,27389.6200,0.3130,1.3980,8174.2900,9999.9900
17,tier,tier5@4-0,val,5463,27389.6200,0.3130,1.3980,8174.2900,9999.9900
15,tier,tier5@2-0,val,5463,27389.6200,0.3130,1.3980,8174.2900,9999.9900
13,tier,tier4@8-0,val,5473,26766.1800,0.3036,1.3790,8185.6700,10000.0000
11,tier,tier4@4-0,val,5473,26766.1800,0.3036,1.3790,8185.6700,10000.0000
9,tier,tier4@2-0,val,5473,26766.1800,0.3036,1.3790,8185.6700,10000.0000


In [34]:
print("Selected per scheme (by val Sharpe):")
for key, (name, params, val_sharpe) in best_per_scheme.items():
    print(f"  {key:10s} -> {name:>22s}  val_sharpe={val_sharpe:.3f}")

best_name = max(
    best_per_scheme.values(), key=lambda x: x[2]
)[0]
print(f"\nBest val config overall: {best_name}")


Selected per scheme (by val Sharpe):
  kelly      ->                kelly@2  val_sharpe=1.144
  tier       ->              tier3@2-0  val_sharpe=1.574
  uniform    ->            uniform@0.5  val_sharpe=1.576
  copy_all   ->               copy_all  val_sharpe=1.334

Best val config overall: uniform@0.5


## Test: single pass per chosen config

One honest test pass for each scheme's val-chosen config (10bps).

In [35]:
for key, (name, params, _val_sharpe) in best_per_scheme.items():
    alpha_map = schemes[name][1]
    res = run_sim(splits["test"], alpha_map, COST_SEL)
    row = sim_row(schemes[name][0], name, "test", res)
    sim_rows.append(row)

sim_df = pd.DataFrame(sim_rows)
sim_df.to_csv(OUT_DIR / "wallet_scaling_sim.csv", index=False)
sim_df[sim_df["split"] == "test"].sort_values("sharpe_daily", ascending=False)


,scheme,config,split,trades,pnl,roi_w,sharpe_daily,mean_used,peak_used
27,tier,tier3@2-0,test,1155,16971.5400,0.3663,1.4360,2254.9700,10000.0000
29,copy_all,copy_all,test,3572,8958.5900,0.1245,0.7450,2719.5100,10000.0000
26,kelly,kelly@2,test,3608,11859.7400,0.1771,0.7000,2554.6300,10000.0000
28,uniform,uniform@0.5,test,3906,12128.8400,0.1880,0.6090,2204.4300,10000.0000


## Robustness: cost sweep + bootstrap CI

Cost sweep (0/10/30bps) + 7-day block-bootstrap Sharpe CI on test.

In [36]:
ci_rows = []
for key, (name, params, _) in best_per_scheme.items():
    alpha_map = schemes[name][1]
    for cost in (0.0, 10.0, 30.0):
        res = run_sim(splits["test"], alpha_map, cost)
        point, lo, hi = block_bootstrap_sharpe(res["daily_pnl"], block_size=7, n_iter=1000, seed=42)
        ci_rows.append({
            "design": name, "cost_bps": cost,
            "pnl": round(res["net_pnl"], 2),
            "roi_w": round(res["net_pnl"] / res["notional"], 4) if res["notional"] > 0 else np.nan,
            "sharpe_daily": round(sizing_sharpe(res["daily_pnl"], 365.0), 3),
            "ci_lo": round(lo, 3), "ci_hi": round(hi, 3),
        })

res_all = capital_constrained_sim(splits["test"], "score1", BUDGET, 1.0, cost_bps=COST_SEL)
point, lo, hi = block_bootstrap_sharpe(res_all["daily_pnl"], block_size=7, n_iter=1000, seed=42)
ci_rows.append({
    "design": "copy_all", "cost_bps": COST_SEL,
    "pnl": round(res_all["net_pnl"], 2),
    "roi_w": round(res_all["net_pnl"] / res_all["notional"], 4) if res_all["notional"] > 0 else np.nan,
    "sharpe_daily": round(sizing_sharpe(res_all["daily_pnl"], 365.0), 3),
    "ci_lo": round(lo, 3), "ci_hi": round(hi, 3),
})

ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(OUT_DIR / "wallet_scaling_ci.csv", index=False)
ci_df


,design,cost_bps,pnl,roi_w,sharpe_daily,ci_lo,ci_hi
0,kelly@2,0.0000,11926.7100,0.1781,0.7000,-0.7050,2.7440
1,kelly@2,10.0000,11859.7400,0.1771,0.7000,-0.7050,2.7440
2,kelly@2,30.0000,11725.7900,0.1751,0.7000,-0.7050,2.7440
3,tier3@2-0,0.0000,17017.8800,0.3673,1.4360,-2.4080,2.1900
4,tier3@2-0,10.0000,16971.5400,0.3663,1.4360,-2.4080,2.1900
5,tier3@2-0,30.0000,16878.8800,0.3643,1.4360,-2.4080,2.1900
6,uniform@0.5,0.0000,12193.3600,0.1890,0.6090,-1.8620,1.7480
7,uniform@0.5,10.0000,12128.8400,0.1880,0.6090,-1.8620,1.7480
8,uniform@0.5,30.0000,11999.7900,0.1860,0.6090,-1.8620,1.7480
9,copy_all,0.0000,9030.5300,0.1255,0.7450,-1.7530,2.5200


## Per-wallet contributions

Train alphas vs forward (test) wallet stats.

In [41]:
test_daily = wallet_daily_pnl(splits["test"])
test_st = test_daily.groupby("wallet")["copyable_pnl"].agg(
    test_pnl="sum", test_n_days="size"
)
test_sharpe = test_daily.groupby("wallet")["copyable_pnl"].apply(
    lambda s: (s.mean() / s.std() * np.sqrt(365.0)) if s.std() > 0 and len(s) >= 2 else np.nan
).rename("test_sharpe")

contrib = st.join(test_st, how="outer").join(test_sharpe, how="outer").fillna(0.0)
contrib = contrib[contrib["test_n_days"] > 0]
alpha_cont = schemes[best_per_scheme["kelly"][0]][1]
alpha_tier_cont = schemes[best_per_scheme["tier"][0]][1]
contrib["alpha_kelly"] = contrib.index.map(alpha_cont).fillna(1.0)
contrib["alpha_tier"] = contrib.index.map(alpha_tier_cont).fillna(1.0)
contrib = contrib.reset_index()
contrib["test_roi"] = contrib["test_pnl"] / contrib["total_pnl"].replace(0, np.nan)
contrib.to_csv(OUT_DIR / "wallet_scaling_contrib.csv", index=False)

a = contrib["alpha_kelly"].to_numpy()
ts = contrib["test_sharpe"].to_numpy()
valid = np.isfinite(ts)
rho = spearman_rho(pd.Series(a[valid]), pd.Series(ts[valid])) if valid.sum() > 2 else np.nan
print(f"Spearman(alpha_kelly, wallet test sharpe) = {rho:.4f}  (n={int(valid.sum())})")
contrib[["wallet", "alpha_kelly", "alpha_tier", "test_pnl", "test_sharpe", "test_roi"]].sort_values('test_pnl', key=abs, ascending=False).head(15)


Spearman(alpha_kelly, wallet test sharpe) = 0.1960  (n=29)


,wallet,alpha_kelly,alpha_tier,test_pnl,test_sharpe,test_roi
10,0x57c5de7efafb020e589f80e0da6e16e9b5c907aa,1.7860,2.0000,32886.2626,7.3584,5.0994
4,0x183a2a0b034877e761af0778da5134bebc37d514,0.8002,2.0000,21905.0395,6.2574,0.5828
23,0xe1b361d6a6f237b9ed7534d19b232df8369e1426,1.0640,0.0000,7837.6638,2.6291,2.9595
15,0x8b72cb885a6bd4ea9d1da393ca231f0fa3476dbe,0.7153,0.0000,4270.2658,1.2055,4.3928
0,0x0cb10c40b0776e9ee8cef970af85724654dda76c,0.6451,0.0000,-3259.1927,-0.6670,-4.6015
25,0xe471565da86af7b6e99d9c955b0b35d1f0e0c87c,1.7197,2.0000,3076.2359,2.9262,0.6061
24,0xe27757cc1dff108d82cb2ce371a40ef7a52b8e46,0.6671,1.0000,2790.8571,5.3824,3.3928
27,0xf72044fb3f98854411a069d9a682e54e4b824599,1.6083,2.0000,2626.2888,10.3325,0.1919
5,0x1c1e841584db14084e10e7dca2ad0ab7b60dbfe7,1.4386,1.0000,-1928.7526,-4.6179,-0.4539
1,0x0e3226217752d7247c67b870b72b99ba3e20535b,0.7620,1.0000,-672.8597,-1.1567,-0.5952


## Save stage 1 result

In [38]:
import json
from datetime import datetime, timezone

best_name = max(best_per_scheme.values(), key=lambda x: x[2])[0]
best_params = schemes[best_name][2]

wallet_cols = [
    "wallet", "mu", "sigma", "n_days", "total_pnl", "sharpe_proxy",
    "alpha_kelly", "alpha_tier", "test_pnl", "test_n_days", "test_sharpe", "test_roi",
]
wallet_records = contrib[[c for c in wallet_cols if c in contrib.columns]].to_dict(orient="records")


def _convert(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


wallet_records = [{k: _convert(v) for k, v in w.items()} for w in wallet_records]

test_row = sim_df[(sim_df["split"] == "test") & (sim_df["config"] == best_name)].iloc[0]
copy_all_row = sim_df[(sim_df["split"] == "test") & (sim_df["config"] == "copy_all")].iloc[0]

metadata = {
    "type": "scaled_copy",
    "tags": sorted(DEFAULT_TAGS),
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "n_wallets_selected": int((contrib["alpha_tier"] > 0).sum()),
    "n_wallets_total": len(wallets),
    "split_sizes": {k: int(len(v)) for k, v in splits.items()},
}

payload = {
    "stage": 1,
    "best_params": {k: _convert(v) for k, v in best_params.items()},
    "best_val_sharpe": float(max(best_per_scheme.values(), key=lambda x: x[2])[2]),
    "test_performance": {
        "config": best_name,
        "trades": int(test_row["trades"]),
        "pnl": float(test_row["pnl"]),
        "roi_w": float(test_row["roi_w"]),
        "sharpe_daily": float(test_row["sharpe_daily"]),
        "copy_all": {
            "trades": int(copy_all_row["trades"]),
            "pnl": float(copy_all_row["pnl"]),
            "roi_w": float(copy_all_row["roi_w"]),
            "sharpe_daily": float(copy_all_row["sharpe_daily"]),
        },
    },
    "metadata": metadata,
    "wallets": wallet_records,
}

out_path = NB_DIR / "stage1_scaled_result.json"
with open(out_path, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved stage 1 scaled result -> {out_path.resolve()}")


Saved stage 1 scaled result -> /Users/vobornij/projects/polymarket/notebooks/wallet_selection/stage1_scaled_result.json


## Price-scaling fill experiment (exploratory)

Test a limit-price entry idea on a **sample** (~1k test contracts, copy-default wallets):
copy each candidate copy-wallet BUY at `limit = price * scale` for
`scale ∈ {1.0, 0.98, 0.95, 0.90}` and give the order a **5-minute window** to fill.

- **Fill rule:** filled iff within `(dt, dt+5min]` any trade on the same
  `(condition_id, token_id)` prints at `price <= limit` with a strictly greater timestamp.
- **Fill price:** exactly the limit price, so
  `pnl = copyable_pnl + copyable_qty * (price - limit)` (same formula/quantity as the
  original `copyable_pnl`); unfilled trades contribute 0.
- **Baseline:** `scale = 1.0` is the market-copy (fill immediately at `price`), so it
  must reproduce `sum(copyable_pnl)` on the sample.

In [7]:
from lib import DEFAULT_TRADES_DIR
from signal_lab.wallet_scaling import price_scale_fill_sim

rng = np.random.RandomState(42)
test_markets = np.sort(splits["test"]["condition_id"].unique())
n_sel = min(1000, len(test_markets))
sel_markets = rng.choice(test_markets, size=n_sel, replace=False)
signals = splits["test"][splits["test"]["condition_id"].isin(sel_markets)].copy()
signals = signals[signals["copyable_qty"] > 0]
print(f"test markets: {len(test_markets):,}  sampled: {n_sel:,}")
print(f"candidate BUYs (copyable_qty>0) on sample: {len(signals):,}")

_tape_cols = ["condition_id", "token_id", "dt", "avg_price"]
tape_parts = []
for f in sorted(DEFAULT_TRADES_DIR.glob("*.parquet")):
    tp = pd.read_parquet(f, columns=_tape_cols)
    tp = tp[tp["condition_id"].isin(sel_markets)]
    if not tp.empty:
        tape_parts.append(tp.rename(columns={"avg_price": "price"}))
tape = (
    pd.concat(tape_parts, ignore_index=True)
    if tape_parts
    else pd.DataFrame(columns=["condition_id", "token_id", "dt", "price"])
)
print(f"fill tape rows (sampled contracts, both sides): {len(tape):,}")


test markets: 1,606  sampled: 1,000
candidate BUYs (copyable_qty>0) on sample: 3,227


fill tape rows (sampled contracts, both sides): 2,221,512


In [8]:
SCALES = (1.0, 0.98, 0.95, 0.90)
sim = price_scale_fill_sim(signals, tape, scales=SCALES, window_minutes=5.0)
base_pnl = float(signals["copyable_pnl"].sum())

summary = (
    sim.groupby("scale")
    .agg(signals=("filled", "size"), fills=("filled", "sum"),
         fill_rate=("filled", "mean"), pnl=("pnl", "sum"))
    .reset_index()
)
summary["pnl_pct_of_market"] = summary["pnl"] / base_pnl * 100 if base_pnl else np.nan
summary["delta_vs_market"] = summary["pnl"] - base_pnl
print(f"market-copy pnl (baseline = sum copyable_pnl): {base_pnl:,.2f}")
summary.round(2)


market-copy pnl (baseline = sum copyable_pnl): 30,358.85


,scale,signals,fills,fill_rate,pnl,pnl_pct_of_market,delta_vs_market
0,0.9000,3227,480,0.1500,13651.1100,44.9700,-16707.7400
1,0.9500,3227,725,0.2200,17555.6100,57.8300,-12803.2500
2,0.9800,3227,968,0.3000,27172.6500,89.5000,-3186.2000
3,1.0000,3227,3227,1.0000,30358.8500,100.0000,0.0000


In [9]:
pw_pnl = sim.pivot_table(index="wallet", columns="scale", values="pnl", aggfunc="sum")
pw_fill = sim.pivot_table(index="wallet", columns="scale", values="filled", aggfunc="mean")
pw = pw_pnl.join(pw_fill.rename(columns={c: f"fill_{c:g}" for c in pw_fill.columns}))
pw = pw.reindex(pw[1.0].sort_values(ascending=False).index)
pw.round(1).head(15)


scale,0.9000,0.9500,0.9800,1.0000,fill_0.9,fill_0.95,fill_0.98,fill_1
wallet,,,,,,,,
0x183a2a0b034877e761af0778da5134bebc37d514,9676.4000,13671.2000,19455.4000,21242.5000,0.1000,0.1000,0.1000,1.0000
0xe1b361d6a6f237b9ed7534d19b232df8369e1426,191.8000,787.4000,4818.6000,7117.2000,0.1000,0.2000,0.3000,1.0000
0xf72044fb3f98854411a069d9a682e54e4b824599,0.0000,388.2000,398.9000,2508.4000,0.0000,0.1000,0.1000,1.0000
0x8b72cb885a6bd4ea9d1da393ca231f0fa3476dbe,1698.3000,780.6000,1168.0000,995.3000,0.3000,0.4000,0.5000,1.0000
0xf59fe344fb8af90a14bd6f2fe62430e56dc03be9,-58.1000,266.8000,250.7000,728.2000,0.1000,0.1000,0.2000,1.0000
0x1c1e841584db14084e10e7dca2ad0ab7b60dbfe7,3512.8000,2301.1000,1555.9000,592.2000,0.2000,0.3000,0.3000,1.0000
0x742ff6df87485287bb4db5b0d7fa7af13047d673,8.7000,101.2000,143.9000,478.6000,0.0000,0.2000,0.3000,1.0000
0x2bcd792138a4e184f791b05d9c5c70e3c8f7cbfb,686.6000,473.2000,511.8000,394.5000,0.3000,0.4000,0.5000,1.0000
0x57c5de7efafb020e589f80e0da6e16e9b5c907aa,0.0000,355.2000,208.2000,334.2000,0.0000,0.0000,0.0000,1.0000


In [10]:
sim.to_csv(OUT_DIR / "price_scale_sim.csv", index=False)
summary.round(4).to_csv(OUT_DIR / "price_scale_summary.csv", index=False)
pw.round(2).reset_index().to_csv(OUT_DIR / "price_scale_wallets.csv", index=False)
print("saved -> signal_lab/price_scale_{sim,summary,wallets}.csv")


saved -> signal_lab/price_scale_{sim,summary,wallets}.csv
